# Reading galaxy morphology across two selected catalogues

This notebook follows the scientific questions from the 13 July 2026 research
meeting. It compares robust galaxy-morphology classifications in the matched
catalogues called `highlum` and `highdens`.

The central goal is to understand what the supplied catalogue measurements and
CNN classifications show. A difference between these selected files is a
catalogue association; this notebook does not assume that environment caused
it.

## 1. Research questions

The analysis follows four questions in a deliberate order:

1. Are the two supplied cross-match products structurally suitable for a
   controlled comparison?
2. How many robust early-type (ETG) and late-type (LTG) classifications occur
   in each matched catalogue?
3. How do classification confidence, apparent brightness, image-plane size,
   viewing orientation, and sky position behave in those samples?
4. Which observed differences and unresolved decisions should be discussed
   with the advisor?

The smaller `highlum` file is always validated first. The same operations are
then applied to `highdens`, which keeps the workflow readable while avoiding a
special analysis definition for either file.

## 2. Data and catalogue language

The morphology definitions come from Vega-Ferrero et al. (2021), especially
Table 6 of *Pushing Automated Morphological Classifications to Their Limits
with the Dark Energy Survey*.

| Column | Meaning in this notebook |
| --- | --- |
| `FLAG_LTG == 4` | robust ETG classification |
| `FLAG_LTG == 5` | robust LTG classification |
| `MP_LTG` | median of the five CNN probabilities of being LTG |
| `MP_EdgeOn` | median of the five CNN probabilities of being edge-on |
| `MAG_AUTO_R` | observed apparent magnitude in the DES r band |
| `FLUX_RADIUS_R` | half-light radius measured in image pixels |
| `Separation` | angular distance between the two matched positions, in arcsec |

Flags 0-3 remain visible as excluded catalogue outcomes, but they are not
relabelled by even/odd parity and do not enter the primary robust comparison.
The probabilities are model outputs rather than independent truth labels.
Likewise, apparent magnitude is not absolute luminosity, and a radius in pixels
is not a physical galaxy size.

## 3. Reproducible setup

The setup below holds every run-level decision in one place. Numerical
statistics use all valid rows. The seed affects only deterministic rendering
samples and the parent-catalogue extraction. Generated results go below the
ignored `outputs/meeting-2026-07-13/` directory.

In [ ]:
# Establish one reproducible configuration without a machine-specific path.
import json
import os
import platform
import subprocess
import sys
from dataclasses import asdict
from datetime import datetime, timezone
from pathlib import Path

candidate_roots = (
    [Path(os.environ["GALAXY_PROJECT_ROOT"])]
    if "GALAXY_PROJECT_ROOT" in os.environ
    else [Path.cwd(), *Path.cwd().parents]
)
PROJECT_ROOT = next(
    (
        candidate.resolve()
        for candidate in candidate_roots
        if (candidate / "requirements.txt").is_file()
        and (candidate / "data" / "README.md").is_file()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Project root requires requirements.txt and data/README.md"
    )
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import astropy
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
from astropy.table import Table
from IPython.display import Markdown, display

from src.galaxy_analysis.artifacts import save_figure, write_json, write_text
from src.galaxy_analysis.catalog import inspect_catalog, random_indices, read_columns
from src.galaxy_analysis.plotting import (
    plot_catalogue_composition,
    plot_magnitude_radius_comparison,
    plot_model_disagreement_comparison,
    plot_morphology_brightness_size_comparison,
    plot_orientation_comparison,
    plot_probability_faintness_comparison,
    plot_robust_probability_comparison,
    plot_separation_comparison,
    plot_sky_footprint_comparison,
)
from src.galaxy_analysis.reporting import (
    brightness_size_class_interpretation,
    composition_interpretation,
    dispersion_interpretation,
    faintness_interpretation,
    magnitude_radius_interpretation,
    orientation_interpretation,
    probability_interpretation,
    radius_cut_interpretation,
    separation_interpretation,
    sky_interpretation,
)
from src.galaxy_analysis.selection import (
    robust_masks,
    valid_value_mask,
    validate_separation,
)
from src.galaxy_analysis.statistics import (
    binned_quantiles,
    compare_catalog_summaries,
    composition_rows,
    describe_values,
    model_dispersion,
    threshold_rows,
)

SEED = 20260713
MAX_SEPARATION_ARCSEC = 1.0
FLUX_RADIUS_CUT = 50.0
PLOT_SAMPLE_SIZE = 100_000
OUTPUT_ROOT = PROJECT_ROOT / "outputs" / "meeting-2026-07-13"
FIGURE_ROOT = OUTPUT_ROOT / "figures"
TABLE_ROOT = OUTPUT_ROOT / "tables"
FIGURE_ROOT.mkdir(parents=True, exist_ok=True)
TABLE_ROOT.mkdir(parents=True, exist_ok=True)

CATALOG_PATHS = {
    "parent_morphology": (
        PROJECT_ROOT / "data" / "DES_DR1_CNN_morphological_catalog.fit"
    ),
    "highlum_source": (
        PROJECT_ROOT
        / "data"
        / "raw"
        / "red-sequence"
        / "redspell_highlum7_final.fits"
    ),
    "highdens_source": (
        PROJECT_ROOT
        / "data"
        / "raw"
        / "red-sequence"
        / "redspell_highdens7_final.fits"
    ),
    "highlum": (
        PROJECT_ROOT
        / "data"
        / "processed"
        / "crossmatches"
        / "vega-ferrero"
        / "match_VF_highlum.fits"
    ),
    "highdens": (
        PROJECT_ROOT
        / "data"
        / "processed"
        / "crossmatches"
        / "vega-ferrero"
        / "match_VF_highdens.fits"
    ),
}

CORE_COLUMNS = (
    "COADD_OBJECT_ID",
    "object_id",
    "RA_2",
    "DEC_2",
    "MAG_AUTO_R",
    "FLUX_RADIUS_R",
    "P1_LTG",
    "P2_LTG",
    "P3_LTG",
    "P4_LTG",
    "P5_LTG",
    "MP_LTG",
    "P1_EdgeOn",
    "P2_EdgeOn",
    "P3_EdgeOn",
    "P4_EdgeOn",
    "P5_EdgeOn",
    "MP_EdgeOn",
    "FLAG_LTG",
    "FLAG_EdgeOn",
    "Separation",
)

saved_figures = []

## 4. Can the catalogues be compared safely?

Opening a FITS table is not enough to trust it. The following gate checks
the required schema, coordinate and probability domains, morphology flags,
duplicate identifiers, and the one-arcsecond angular match rule. It validates
`highlum` first and confirms its meeting-derived baseline before loading
`highdens`.

The concise table is meant for discussion. Complete machine-readable details
are saved locally in the output directory.

In [ ]:
# Validate highlum first, then apply the identical quality gate to highdens.
schemas = {
    alias: inspect_catalog(path)
    for alias, path in CATALOG_PATHS.items()
}
catalogues = {}
separation_checks = {}
quality_rows = []

probability_columns = tuple(
    name
    for name in CORE_COLUMNS
    if name.startswith("P") or name.startswith("MP_")
)

for alias in ("highlum", "highdens"):
    data = read_columns(CATALOG_PATHS[alias], CORE_COLUMNS)
    schema = schemas[alias]
    separation_check = validate_separation(
        data["Separation"],
        MAX_SEPARATION_ARCSEC,
    )
    invalid_coordinates = int(
        np.count_nonzero(~valid_value_mask(data["RA_2"], "ra_deg"))
        + np.count_nonzero(~valid_value_mask(data["DEC_2"], "dec_deg"))
    )
    invalid_probabilities = int(
        sum(
            np.count_nonzero(
                ~valid_value_mask(data[column], "probability")
            )
            for column in probability_columns
        )
    )
    invalid_flags = int(
        np.count_nonzero(~np.isin(data["FLAG_LTG"], np.arange(6)))
    )
    duplicate_morphology_ids = int(
        len(data["COADD_OBJECT_ID"])
        - len(np.unique(data["COADD_OBJECT_ID"]))
    )
    duplicate_environment_ids = int(
        len(data["object_id"]) - len(np.unique(data["object_id"]))
    )

    if schema.row_count != len(data["FLAG_LTG"]):
        raise ValueError(f"{alias} row count changed while reading")
    if not separation_check.is_valid:
        raise ValueError(f"{alias} contains matches outside 1 arcsec")
    if invalid_coordinates or invalid_probabilities or invalid_flags:
        raise ValueError(f"{alias} failed value-domain validation")

    catalogues[alias] = data
    separation_checks[alias] = separation_check
    quality_rows.append(
        (
            alias,
            schema.row_count,
            schema.column_count,
            separation_check.invalid_count,
            invalid_coordinates,
            invalid_probabilities,
            invalid_flags,
            duplicate_morphology_ids,
            duplicate_environment_ids,
            "PASS",
        )
    )

expected_highlum_flags = {
    0: 5_220,
    1: 3_039,
    2: 344,
    3: 404,
    4: 24_871,
    5: 890,
}
observed_flags, observed_counts = np.unique(
    catalogues["highlum"]["FLAG_LTG"],
    return_counts=True,
)
observed_highlum_flags = {
    int(flag): int(count)
    for flag, count in zip(observed_flags, observed_counts, strict=True)
}
assert schemas["highlum"].row_count == 34_768
assert schemas["highdens"].row_count == 905_291
assert observed_highlum_flags == expected_highlum_flags

quality_table = Table(
    rows=quality_rows,
    names=(
        "catalogue",
        "rows",
        "available_columns",
        "invalid_separations",
        "invalid_coordinates",
        "invalid_probabilities",
        "invalid_flags",
        "duplicate_morphology_ids",
        "duplicate_environment_ids",
        "status",
    ),
)
quality_table.write(
    TABLE_ROOT / "catalogue_quality.csv",
    format="ascii.csv",
    overwrite=True,
)
write_json(
    OUTPUT_ROOT / "catalogue_quality.json",
    {
        row["catalogue"]: {
            name: row[name]
            for name in quality_table.colnames
            if name != "catalogue"
        }
        for row in quality_table
    },
)
display(quality_table)

### What the quality gate establishes

Both supplied products satisfy the adopted coordinate, probability, flag, and
one-arcsecond match rules. The exact `highlum` flag counts reproduce the
meeting-derived baseline, so the comparison proceeds without silently changing
the robust sample definition. Duplicate counts remain visible because repeated
identifiers would change how independent rows should be interpreted.

### Careful interpretation

Passing these checks establishes internal catalogue consistency, not scientific
completeness. The RA/DEC FITS units are not declared, and the physical meanings
of the `highlum` and `highdens` source selections still require confirmation.

In [ ]:
# Calculate reusable full-row summaries once for every later figure.
masks_by_catalogue = {
    alias: robust_masks(data["FLAG_LTG"])
    for alias, data in catalogues.items()
}
composition_by_catalogue = {
    alias: composition_rows(alias, data["FLAG_LTG"])
    for alias, data in catalogues.items()
}
thresholds_by_catalogue = {
    alias: threshold_rows(alias, data["MP_LTG"], data["FLAG_LTG"])
    for alias, data in catalogues.items()
}

summary_rows = []
dispersion_by_catalogue = {}
summary_variables = (
    "MP_LTG",
    "MP_EdgeOn",
    "MAG_AUTO_R",
    "FLUX_RADIUS_R",
    "Separation",
)

for alias, data in catalogues.items():
    masks = masks_by_catalogue[alias]
    samples = {
        "all_valid": np.ones(len(data["FLAG_LTG"]), dtype=bool),
        "robust_etg": masks.robust_etg,
        "robust_ltg": masks.robust_ltg,
    }
    for class_name, sample_mask in samples.items():
        for variable in summary_variables:
            summary_rows.append(
                describe_values(
                    alias,
                    class_name,
                    variable,
                    data[variable][sample_mask],
                )
            )

    model_probabilities = np.column_stack(
        [data[f"P{model}_LTG"] for model in range(1, 6)]
    )
    dispersion = model_dispersion(model_probabilities)["std_ddof1"]
    dispersion_by_catalogue[alias] = {
        "robust_etg": dispersion[masks.robust_etg],
        "robust_ltg": dispersion[masks.robust_ltg],
    }
    for class_name in ("robust_etg", "robust_ltg"):
        summary_rows.append(
            describe_values(
                alias,
                class_name,
                "P1_P5_LTG_STD",
                dispersion_by_catalogue[alias][class_name],
            )
        )
    del model_probabilities, dispersion

composition_table = Table(
    rows=[
        asdict(row)
        for alias in ("highlum", "highdens")
        for row in composition_by_catalogue[alias]
    ]
)
summary_table = Table(rows=[asdict(row) for row in summary_rows])
threshold_table = Table(
    rows=[
        asdict(row)
        for alias in ("highlum", "highdens")
        for row in thresholds_by_catalogue[alias]
    ]
)
for filename, table in (
    ("morphology_composition.csv", composition_table),
    ("summary_statistics.csv", summary_table),
    ("threshold_sensitivity.csv", threshold_table),
):
    table.write(TABLE_ROOT / filename, format="ascii.csv", overwrite=True)

display(composition_table)
display(
    summary_table[
        "catalog",
        "class_name",
        "variable",
        "n_valid",
        "median",
        "iqr",
    ]
)

## 5. Which morphologies dominate the samples?

The first figure separates robust ETGs, robust LTGs, and every flag
excluded from the primary analysis. Counts show the available sample size;
fractions allow comparison despite the much larger `highdens` file.

In [ ]:
# Compare robust-class composition before interpreting other distributions.
composition_figure = plot_catalogue_composition(
    composition_by_catalogue["highlum"],
    composition_by_catalogue["highdens"],
)
saved_figures.append(
    save_figure(
        composition_figure,
        FIGURE_ROOT,
        "01_catalogue_composition.png",
    )
)
display(composition_figure)
plt.close(composition_figure)

In [ ]:
# Explain the measured composition using the same rows shown in Figure 1.
display(
    Markdown(
        composition_interpretation(
            composition_by_catalogue["highlum"],
            composition_by_catalogue["highdens"],
        )
    )
)

### Careful interpretation

A dominant robust ETG fraction is not, by itself, evidence that the classifier
failed. These source catalogues were selected before the morphology match, so
their luminosity, density, red-sequence, magnitude, and redshift mixtures can
strongly shape the observed class proportions. Wilson intervals describe
counting precision but do not include those selection uncertainties.

## 6. How confident are the robust classifications?

`FLAG_LTG` defines the selected robust class, whereas `MP_LTG` is the
median continuous output of five CNN models. Normalized distributions reveal
where each class lies on the probability scale without allowing the larger ETG
sample to dominate by raw count.

In [ ]:
# Compare MP_LTG shapes within each robust class on shared probability axes.
probability_values = {
    alias: {
        "robust_etg": data["MP_LTG"][masks_by_catalogue[alias].robust_etg],
        "robust_ltg": data["MP_LTG"][masks_by_catalogue[alias].robust_ltg],
    }
    for alias, data in catalogues.items()
}
probability_figure = plot_robust_probability_comparison(
    probability_values["highlum"],
    probability_values["highdens"],
)
saved_figures.append(
    save_figure(
        probability_figure,
        FIGURE_ROOT,
        "02_robust_ltg_probability.png",
    )
)
display(probability_figure)
plt.close(probability_figure)

In [ ]:
# Report class medians and IQRs rather than relying on visual separation.
display(Markdown(probability_interpretation(summary_rows)))
display(threshold_table)

### Careful interpretation

The strong separation is expected because the robust flags were built from the
same five model predictions. Thresholds 0.5, 0.6, and 0.8 are shown only as
internal sensitivity checks against flag 5; they do not replace the robust flag
definition or provide independent validation against true morphology.

Two objects can have the same median `MP_LTG` while their five networks
agree very differently. The next graph therefore compares the row-wise sample
standard deviation of P1-P5 within the robust classes.

In [ ]:
# Compare five-model dispersion separately from median classification output.
disagreement_figure = plot_model_disagreement_comparison(
    dispersion_by_catalogue["highlum"],
    dispersion_by_catalogue["highdens"],
)
saved_figures.append(
    save_figure(
        disagreement_figure,
        FIGURE_ROOT,
        "03_model_disagreement.png",
    )
)
display(disagreement_figure)
plt.close(disagreement_figure)

In [ ]:
# Identify the measured class with the largest typical model disagreement.
display(Markdown(dispersion_interpretation(summary_rows)))

### Careful interpretation

P1-P5 dispersion is a useful internal diagnostic, but the networks share a
training strategy and may share biases. This statistic does not include image
depth, seeing, catalogue selection, incorrect training labels, or domain shift.

## 7. Where do the galaxies lie in brightness-size space?

The next density panels use every finite row to show where each matched
catalogue is concentrated in apparent magnitude and half-light radius. Shared
axes prevent autoscaling from manufacturing a visual difference, and the
magnitude direction is reversed so brighter objects lie to the left.

In [ ]:
# Show the complete brightness-radius density before selecting morphology.
magnitude_radius_figure = plot_magnitude_radius_comparison(
    catalogues["highlum"],
    catalogues["highdens"],
)
saved_figures.append(
    save_figure(
        magnitude_radius_figure,
        FIGURE_ROOT,
        "04_magnitude_half_light_radius.png",
    )
)
display(magnitude_radius_figure)
plt.close(magnitude_radius_figure)

In [ ]:
# Describe measured ranges and audit the provisional 50-pixel cut.
display(Markdown(magnitude_radius_interpretation(summary_rows)))
display(
    Markdown(
        radius_cut_interpretation(
            catalogues["highlum"]["FLUX_RADIUS_R"],
            catalogues["highdens"]["FLUX_RADIUS_R"],
            FLUX_RADIUS_CUT,
        )
    )
)

### Careful interpretation

The graph relates two observed image quantities. Apparent magnitude mixes
intrinsic luminosity and distance, while `FLUX_RADIUS_R` is measured in pixels.
Seeing, crowding, surface-brightness limits, and catalogue selection can shape
the dense locus. The provisional 50-pixel rule is reported but is not applied
to the primary statistics.

The all-row density can hide differences between robust morphology
classes. Four small multiples now hold catalogue and class separate, using the
same axes so their locations and spreads can be compared directly.

In [ ]:
# Compare robust-class density without letting ETG abundance set point opacity.
morphology_size_figure = plot_morphology_brightness_size_comparison(
    catalogues["highlum"],
    masks_by_catalogue["highlum"],
    catalogues["highdens"],
    masks_by_catalogue["highdens"],
)
saved_figures.append(
    save_figure(
        morphology_size_figure,
        FIGURE_ROOT,
        "05_morphology_brightness_size.png",
    )
)
display(morphology_size_figure)
plt.close(morphology_size_figure)

In [ ]:
# Quantify the class radius contrast shown by the four density panels.
display(Markdown(brightness_size_class_interpretation(summary_rows)))

### Careful interpretation

The class panels can reveal different observed loci, but `MAG_AUTO_R`,
`FLUX_RADIUS_R`, and the CNN classifications all come from related imaging.
Differences may therefore reflect shared measurement quality or sample
selection rather than a separate physical morphology-size relation.

## 8. Does classification structure change for fainter galaxies?

The paper explicitly examines whether model performance changes for
fainter galaxies. These matched products have no independent truth labels, so
we ask the narrower question available here: does the distribution of
`MP_LTG` itself change across apparent magnitude?

In [ ]:
# Overlay fixed-bin medians and IQRs on the full probability density.
pooled_magnitude = np.concatenate(
    [
        np.asarray(catalogues[alias]["MAG_AUTO_R"], dtype=float)
        for alias in ("highlum", "highdens")
    ]
)
finite_magnitude = pooled_magnitude[np.isfinite(pooled_magnitude)]
magnitude_bins = np.linspace(
    float(finite_magnitude.min()),
    float(finite_magnitude.max()),
    13,
)
faintness_summaries = {
    alias: binned_quantiles(
        data["MAG_AUTO_R"],
        data["MP_LTG"],
        magnitude_bins,
    )
    for alias, data in catalogues.items()
}
faintness_figure = plot_probability_faintness_comparison(
    catalogues["highlum"],
    catalogues["highdens"],
    magnitude_bins,
)
saved_figures.append(
    save_figure(
        faintness_figure,
        FIGURE_ROOT,
        "06_probability_vs_faintness.png",
    )
)
display(faintness_figure)
plt.close(faintness_figure)
del pooled_magnitude, finite_magnitude

In [ ]:
# State the measured bright-to-faint median change for each catalogue.
display(
    Markdown(
        "\n\n".join(
            faintness_interpretation(alias, faintness_summaries[alias])
            for alias in ("highlum", "highdens")
        )
    )
)

### Careful interpretation

A change in median probability is not a measured change in classification
accuracy. It can also arise because the mix of ETG/LTG flags, redshift, image
quality, or source selection changes with apparent magnitude. Independent
labels would be required to measure accuracy in these matched samples.

## 9. How are morphology and orientation outputs related?

A disk viewed edge-on can hide spiral structure and complicate visual
morphology. The joint probability density therefore shows the morphology and
orientation outputs together while keeping their meanings separate.

In [ ]:
# Compare LTG and edge-on outputs on identical probability domains.
orientation_figure = plot_orientation_comparison(
    catalogues["highlum"],
    catalogues["highdens"],
)
saved_figures.append(
    save_figure(
        orientation_figure,
        FIGURE_ROOT,
        "07_ltg_vs_edgeon_probability.png",
    )
)
display(orientation_figure)
plt.close(orientation_figure)

In [ ]:
# Compare edge-on medians while retaining their orientation meaning.
display(Markdown(orientation_interpretation(summary_rows)))

### Careful interpretation

`MP_EdgeOn` is not a third morphology class. It is a model output about viewing
orientation, and the paper recommends combining it carefully with the ETG/LTG
classification because projection can make disks resemble smoother systems.

## 10. What part of the sky does each sample cover?

The sky map checks whether the two products cover the same observed
regions. Only point rendering is sampled; coordinate ranges and all numerical
summaries continue to use every valid catalogue row.

In [ ]:
# Sample only point rendering so sky holes remain visible without excess memory.
sky_plot_indices = {
    alias: random_indices(
        len(data["RA_2"]),
        PLOT_SAMPLE_SIZE,
        SEED,
    )
    for alias, data in catalogues.items()
}
sky_plot_data = {
    alias: {
        "RA_2": data["RA_2"][sky_plot_indices[alias]],
        "DEC_2": data["DEC_2"][sky_plot_indices[alias]],
    }
    for alias, data in catalogues.items()
}
sky_figure = plot_sky_footprint_comparison(
    sky_plot_data["highlum"],
    sky_plot_data["highdens"],
)
saved_figures.append(
    save_figure(
        sky_figure,
        FIGURE_ROOT,
        "08_sky_footprint.png",
    )
)
display(sky_figure)
plt.close(sky_figure)

In [ ]:
# Report full-catalogue coordinate ranges and the mask interpretation.
display(
    Markdown(
        sky_interpretation(
            catalogues["highlum"]["RA_2"],
            catalogues["highlum"]["DEC_2"],
            catalogues["highdens"]["RA_2"],
            catalogues["highdens"]["DEC_2"],
        )
    )
)

### Careful interpretation

The plot shows catalogue coverage, not a continuous density field. Empty areas
can follow the DES footprint or masks around saturated stars, as discussed in
the meeting. Without the official survey mask, they must not be interpreted as
physical underdensities.

## 11. Are the coordinate matches comparable?

The supplied files were matched with a one-arcsecond angular tolerance.
Normalized logarithmic histograms show the narrow separation core and the tail
without allowing the larger `highdens` catalogue to dominate by count.

In [ ]:
# Compare match separations against the same one-arcsecond rule.
separation_figure = plot_separation_comparison(
    catalogues["highlum"]["Separation"],
    catalogues["highdens"]["Separation"],
    max_arcsec=MAX_SEPARATION_ARCSEC,
)
saved_figures.append(
    save_figure(
        separation_figure,
        FIGURE_ROOT,
        "09_match_separation.png",
    )
)
display(separation_figure)
plt.close(separation_figure)

In [ ]:
# State the median, tail, and adopted angular match rule explicitly.
display(
    Markdown(
        separation_interpretation(
            catalogues["highlum"]["Separation"],
            catalogues["highdens"]["Separation"],
            MAX_SEPARATION_ARCSEC,
        )
    )
)

### Careful interpretation

A small angular separation supports catalogue association but does not prove
that every match is astrophysically correct. One arcsecond is an angle on the
sky, not a physical distance; converting it to physical scale would require
redshift and a cosmological model.

## 12. Conclusions and questions for the next meeting

In [ ]:
# Save one signed highdens-minus-highlum comparison table for review.
comparison_variables = {
    "MP_LTG",
    "MP_EdgeOn",
    "MAG_AUTO_R",
    "FLUX_RADIUS_R",
    "Separation",
}
highlum_summary_rows = [
    row
    for row in summary_rows
    if row.catalog == "highlum" and row.variable in comparison_variables
]
highdens_summary_rows = [
    row
    for row in summary_rows
    if row.catalog == "highdens" and row.variable in comparison_variables
]
comparison_rows = compare_catalog_summaries(
    highlum_summary_rows,
    highdens_summary_rows,
)
comparison_table = Table(rows=[asdict(row) for row in comparison_rows])
comparison_table.write(
    TABLE_ROOT / "highdens_minus_highlum.csv",
    format="ascii.csv",
    overwrite=True,
)
display(comparison_table)

In [ ]:
# Save reproducible parent row positions without copying catalogue values.
parent_sample_directory = OUTPUT_ROOT / "parent-sample"
parent_sample_directory.mkdir(parents=True, exist_ok=True)
parent_indices = random_indices(
    schemas["parent_morphology"].row_count,
    PLOT_SAMPLE_SIZE,
    SEED,
)
np.save(parent_sample_directory / "sample_indices.npy", parent_indices)
assert np.unique(parent_indices).size == parent_indices.size
assert np.all(parent_indices[:-1] < parent_indices[1:])
parent_sample_directory / "sample_indices.npy"

In [ ]:
# Answer the opening questions with values verified in this execution.
highlum_composition = {
    row.class_name: row
    for row in composition_by_catalogue["highlum"]
}
highdens_composition = {
    row.class_name: row
    for row in composition_by_catalogue["highdens"]
}
conclusion_text = f"""### Answers supported by this run

1. **Catalogue quality:** both match products pass the required schema,
   probability, coordinate, flag, identifier, and one-arcsecond checks used in
   this notebook.
2. **Robust composition:** `highlum` contains
   {highlum_composition['robust_etg'].count:,} robust ETG and
   {highlum_composition['robust_ltg'].count:,} robust LTG classifications;
   `highdens` contains {highdens_composition['robust_etg'].count:,} and
   {highdens_composition['robust_ltg'].count:,}, respectively.
3. **Model and observable structure:** the figures quantify how probabilities,
   five-model disagreement, apparent magnitude, half-light radius in pixels,
   orientation output, sky coverage, and match separation differ between the
   selected files.
4. **Scientific conclusion:** these are descriptive catalogue associations.
   Source selection, distance mixture, image quality, projection, and masks
   prevent a causal environmental interpretation at this stage.

All nine figures and the supporting tables are saved under
`outputs/meeting-2026-07-13/` and can be regenerated from this notebook.
"""
advisor_summary_path = write_text(
    OUTPUT_ROOT / "advisor_summary.md",
    conclusion_text,
)
display(Markdown(conclusion_text))

### Questions for the next meeting

1. What physical source-selection rules do `highlum` and `highdens` represent?
2. Was `FLUX_RADIUS_R < 50` intended as a filter, an axis limit, or a check on
   the parent morphology catalogue?
3. Should probability thresholds 0.5, 0.6, and 0.8 remain sensitivity checks?
4. Can the corrected catalogue with the missing morphometric parameters from
   Fabrício be provided?
5. Should a later phase convert image-plane radii to physical radii using
   redshift, pixel scale, and cosmology?

The Ferrari et al. morphometric analysis remains deferred until those missing
parameters are available.

In [ ]:
# Record code, software, input provenance, parameters, and generated products.
git_result = subprocess.run(
    ["git", "rev-parse", "HEAD"],
    cwd=PROJECT_ROOT,
    check=False,
    capture_output=True,
    text=True,
)
git_commit = (
    git_result.stdout.strip()
    if git_result.returncode == 0
    else None
)
input_manifest = {}
for alias, path in CATALOG_PATHS.items():
    file_status = path.stat()
    schema = schemas[alias]
    input_manifest[alias] = {
        "path": str(path.relative_to(PROJECT_ROOT)),
        "size_bytes": file_status.st_size,
        "modified_utc": datetime.fromtimestamp(
            file_status.st_mtime,
            timezone.utc,
        ).isoformat(),
        "table_hdu": schema.hdu_index,
        "rows": schema.row_count,
        "columns": schema.column_count,
    }

generated_product_paths = [
    OUTPUT_ROOT / "catalogue_quality.json",
    OUTPUT_ROOT / "advisor_summary.md",
    TABLE_ROOT / "catalogue_quality.csv",
    TABLE_ROOT / "morphology_composition.csv",
    TABLE_ROOT / "threshold_sensitivity.csv",
    TABLE_ROOT / "summary_statistics.csv",
    TABLE_ROOT / "highdens_minus_highlum.csv",
    parent_sample_directory / "sample_indices.npy",
    *saved_figures,
]
assert all(path.is_file() for path in generated_product_paths)
generated_products = sorted(
    str(path.relative_to(OUTPUT_ROOT))
    for path in generated_product_paths
)
run_metadata = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "software": {
        "python": platform.python_version(),
        "numpy": np.__version__,
        "astropy": astropy.__version__,
        "matplotlib": matplotlib.__version__,
    },
    "git_commit": git_commit,
    "inputs": input_manifest,
    "parameters": {
        "seed": SEED,
        "plot_sample_size_maximum": PLOT_SAMPLE_SIZE,
        "parent_sample_size": int(parent_indices.size),
        "maximum_separation_arcsec": MAX_SEPARATION_ARCSEC,
        "provisional_flux_radius_cut": FLUX_RADIUS_CUT,
        "robust_etg_rule": "FLAG_LTG == 4",
        "robust_ltg_rule": "FLAG_LTG == 5",
    },
    "generated_products": generated_products,
    "quality_warnings": [
        "RA_2 and DEC_2 lack declared FITS units; degrees are assumed",
        "CNN probabilities are not independent truth labels",
        "highlum and highdens selection meanings await confirmation",
        "FLUX_RADIUS_R is an image radius in pixels, not physical size",
    ],
}
metadata_path = write_json(
    OUTPUT_ROOT / "run_metadata.json",
    run_metadata,
)
display(Markdown(f"Run metadata saved to `{metadata_path}`."))